# <font color="steelblue">19. Tendencia con medias móviles y desestacionalización</font>

*Unidad 3. Números índice y series cronológicas* · Sesión teórica 19 de 30

**Autoría**:

*   Fernando Borrás Rocher (f.borras@umh.es), autor de la idea original
*   Juan Francisco Monge Ivars
*   José Santiago García Cremades

Departamento de Estadística, Matemáticas e Informática. Universidad Miguel Hernández de Elche.

Basado en [SIMUSTAT](https://unidigitalsimustat.umh.es/) (proyecto UNIDIGITAL-UMH, CC BY-SA 4.0).

**Asignatura**: Estadística (1200), 1.º curso, Grado en Periodismo.

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="CC BY-SA 4.0" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a> Creative Commons Atribución-CompartirIgual 4.0.

> ⚠️ **Los datos de este curso son simulados.** Imitan la forma de fuentes reales (INE, CIS, audímetros, agencias de verificación), pero ninguna cifra es real. Úsalos para aprender; para publicar, ve siempre a la fuente oficial.

## <font color='steelblue'>Antes de empezar: configura el cuaderno</font>

Ejecuta **una vez** la celda de abajo (botón ▶). Instala la librería de las preguntas, descarga los datos (`paro_mensual.csv`, `visitas_diario.csv`) y las preguntas de autoevaluación. Si todo va bien verás **`Cuaderno configurado`**.

> 💡 Si Colab se desconecta, los ficheros se borran: vuelve a ejecutar esta celda.

In [ ]:
#@title <font color="steelblue" size="+1"> Ejecutar configuración { display-mode: "form" }
import os, sys, json, base64, subprocess, urllib.request

RAW = "https://raw.githubusercontent.com/fborrasumh/statperiodismo/main/"
DATOS = ["paro_mensual.csv", "visitas_diario.csv"]
PREGUNTAS = ["q_P19_1.json", "q_P19_2.json", "q_P19_3.json"]

try:
    from jupyterquiz import display_quiz
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupyterquiz==2.9.6.4"], check=True)
    from jupyterquiz import display_quiz

for carpeta, ficheros in (("datos/", DATOS), ("autoeval/", PREGUNTAS)):
    for f in ficheros:
        if not os.path.exists(f):
            urllib.request.urlretrieve(RAW + carpeta + f, f)

def pregunta(fichero):
    """Muestra una pregunta interactiva de autoevaluación."""
    display_quiz(json.loads(base64.b64decode(open(fichero, "rb").read()).decode("utf8")))

print("Cuaderno configurado")

## <font color='steelblue'>Introducción</font>

Los organismos estadísticos publican a menudo dos cifras: la «original» y la «desestacionalizada». La segunda quita el efecto de la temporada para ver si la situación mejora o empeora de verdad. Aquí aprenderás la versión sencilla de ese proceso: **medias móviles** para la tendencia e **índices de variación estacional** para medir y quitar la estacionalidad.

**Qué necesitas saber antes**: el cuaderno 18.

## <font color='steelblue'>Qué vas a aprender</font>

Al terminar este cuaderno sabrás:

1. Calcular medias móviles y usarlas para ver la tendencia.
2. Calcular índices de variación estacional.
3. Desestacionalizar una serie y leer la serie corregida.

## <font color='steelblue'>Contenidos</font>

1. Medias móviles
2. Índices de variación estacional
3. Desestacionalizar

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.max_columns', 20)

## <font color='steelblue'>1. Medias móviles</font>

In [ ]:
paro = pd.read_csv("paro_mensual.csv")
paro["fecha"] = pd.to_datetime(paro["fecha"])
paro = paro.set_index("fecha")
visitas = pd.read_csv("visitas_diario.csv")
visitas["fecha"] = pd.to_datetime(visitas["fecha"])
visitas = visitas.set_index("fecha")["visitas"]

Una **media móvil** de orden $k$ sustituye cada valor por la media de $k$ valores consecutivos. Si $k$ coincide con el periodo estacional (7 días, 12 meses), la estacionalidad desaparece y queda la **tendencia**. `rolling(k, center=True)` la calcula centrada.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
visitas.plot(ax=ax, lw=0.5, alpha=0.5, label="Diario")
visitas.rolling(7, center=True).mean().plot(ax=ax, label="Media móvil 7 días")
visitas.rolling(91, center=True).mean().plot(ax=ax, lw=2, label="Media móvil 91 días")
ax.legend()
ax.set_title("Visitas al diario: el ruido y la tendencia (datos simulados)")
plt.show()

**Qué ha pasado**: la media de 7 días quita el efecto del fin de semana; la de 91 días (un trimestre) deja ver la tendencia creciente. Las «medias de 7 días» se popularizaron en prensa durante la pandemia precisamente para eso.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P19_1.json")

## <font color='steelblue'>2. Índices de variación estacional</font>

Los **índices de variación estacional** (IVE) dicen cuánto se aparta cada mes, en promedio, de la tendencia. Método sencillo (de razón a la media móvil):

1. Calcular la tendencia con una media móvil centrada de 12 meses.
2. Dividir cada dato por su tendencia.
3. Promediar esos cocientes para cada mes del año.

In [ ]:
tendencia = paro["parados"].rolling(12, center=True).mean()
razon = paro["parados"] / tendencia
ive = razon.groupby(razon.index.month).mean()
ive = ive / ive.mean() * 100            # que su media sea 100
ive.round(1)

In [ ]:
ive.plot(kind="bar", color=["firebrick" if v > 100 else "steelblue" for v in ive])
plt.axhline(100, color="grey", lw=0.8)
plt.xticks(range(12), ["E", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"], rotation=0)
plt.title("Índice de variación estacional del paro (100 = mes normal)")
plt.show()

**Cómo leerlo**: un IVE de 93,5 en julio significa que en julio el paro suele estar un 6,5 % por debajo de su tendencia. Enero y febrero están por encima (104).

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P19_2.json")

## <font color='steelblue'>3. Desestacionalizar</font>

Para **desestacionalizar** se divide cada dato por el IVE de su mes (y se multiplica por 100):

In [ ]:
paro["desestacionalizado"] = paro["parados"] / paro.index.month.map(ive) * 100
paro[["parados", "desestacionalizado"]].plot(figsize=(11, 4))
plt.title("Paro original y desestacionalizado (datos simulados)")
plt.show()

In [ ]:
ultimos = paro[["parados", "desestacionalizado"]].diff().tail(6).round()
ultimos.columns = ["variación mensual original", "variación mensual desestacionalizada"]
ultimos

**Cómo leerlo**: la serie original baja en verano y sube en otoño; la desestacionalizada muestra si, quitando la temporada, el paro mejora o empeora. Los organismos oficiales usan métodos más sofisticados (como TRAMO-SEATS o X-13), pero la idea es la misma.

> En una noticia: «el paro sube en octubre por el fin de la temporada, pero en términos desestacionalizados baja por quinto mes consecutivo» (si los datos lo dicen).

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P19_3.json")

## <font color='steelblue'>Reto aprendiz</font>

Resuelve estos ejercicios en la celda vacía de debajo de cada enunciado. Si te atascas, vuelve a la sección indicada. La solución está plegada: **inténtalo antes de mirarla**.

### **Ejercicio 1** *(sección 1)*

Calcula y representa la media móvil centrada de 12 meses del paro sobre la serie original.

<details><summary>Ver solución</summary>

```python
paro["parados"].plot()
paro["parados"].rolling(12, center=True).mean().plot()
plt.show()
```
</details>

### **Ejercicio 2** *(sección 2)*

¿Qué mes tiene el IVE más bajo y cuál el más alto?

<details><summary>Ver solución</summary>

```python
ive.idxmin(), ive.idxmax()
```
</details>

### **Ejercicio 3** *(sección 2)*

Calcula los índices de variación estacional **semanal** de las visitas al diario (día de la semana), con una media móvil de 7 días.

<details><summary>Ver solución</summary>

```python
t = visitas.rolling(7, center=True).mean()
r = (visitas / t).groupby(visitas.index.dayofweek).mean()
(r / r.mean() * 100).round(1)
```
</details>

### **Ejercicio 4** *(sección 3)*

Escribe el titular del último mes de la serie usando a la vez el dato original y el desestacionalizado.

<details><summary>Ver idea clave</summary>

Usa la tabla `ultimos`: di qué pasa con la cifra original y qué dice la desestacionalizada.
</details>

## <font color='steelblue'>Resumen: ideas clave</font>

- Una **media móvil** de orden igual al periodo estacional elimina la estacionalidad y deja ver la tendencia.
- Los **índices de variación estacional** miden cuánto se aparta, de media, cada mes de la tendencia (100 = normal).
- **Desestacionalizar**: dato / IVE × 100.
- En prensa, combina el dato original con el **desestacionalizado** para separar temporada y tendencia.

## <font color='steelblue'>Glosario</font>

| Término | Significado |
| :-- | :-- |
| **Media móvil** | Media de k valores consecutivos que se va desplazando. |
| **IVE** | Índice de variación estacional: desviación media de cada periodo respecto a la tendencia. |
| **Serie desestacionalizada** | Serie a la que se ha quitado el componente estacional. |

## <font color='steelblue'>Errores frecuentes y cómo resolverlos</font>

| Síntoma | Causa probable | Solución |
| :-- | :-- | :-- |
| Los primeros y últimos valores de la media móvil son NaN | No hay suficientes datos alrededor | Es normal en las medias centradas |
| `NameError: name 'pd' is not defined` | No se ejecutó la celda de importación | Ejecuta las celdas desde el principio, en orden |
| `FileNotFoundError` al leer un fichero | No se ejecutó la configuración o Colab se reinició | Ejecuta la celda *Configura el cuaderno* |
| La pregunta de autoevaluación no aparece | Falta la configuración | Ejecuta la celda de configuración y luego la pregunta |

## <font color='steelblue'>Para seguir</font>

Con esto termina la unidad 3. En el cuaderno 20 empieza la Probabilidad: azar y simulación.

**Siguiente cuaderno**: [20. Azar y probabilidad: experimentos, sucesos y simulación](https://colab.research.google.com/github/fborrasumh/statperiodismo/blob/main/cuadernos/P20_azar_y_probabilidad.ipynb)